In [3]:
# Importando as bibliotecas
import requests
import pandas as pd
import numpy as np
import re
import os


# * Listando os parâmetros que serão listados na API
parametros = {
    '@trimestre': "'20201'",
    '$top': 1744,
    '$format': 'json',
    '$select': 'trimestre,canalAcesso,produto,acessoATM,qtdTransacoes,valorTransacoes'
}

site = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/TRANSOPADA(trimestre=@trimestre)'


# * Requisição + Tratamento de Erro
try:
    response = requests.get(url=site, params=parametros)
    response.raise_for_status()
    dados = response.json()
    
    # - Salvando os arquivos em um DataFrame
    dados_brutos = dados['value']
    df = pd.DataFrame(dados_brutos)

     # * Tratamento de Dados
    df_copia = df.copy()
    
    
    # Função que insere sublinhado antes de maiúsculas e converte para minúsculas
    def camel_to_snake(name):
        
        # Adiciona '_' antes de maiúsculas e remove espaços extras
        s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
        return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()

    # Aplicando a conversão em todas as colunas
    df_copia.columns = [camel_to_snake(col) for col in df.columns]
    
    
    
    # - Tratamento de dados
    # / Separando o trimestre e o ano em colunas diferentes 
    df_copia['ano'] = df_copia['trimestre'].astype(str).str[-1]
    df_copia['trimestre'] = df_copia['trimestre'].astype(str).str[:4]
    
    # / Alterando o tipo do ano -> int
    df_copia['trimestre'] = df_copia['trimestre'].astype(int)
    df_copia['ano'] = df_copia['ano'].astype(int)
    
    # / Renomeando as colunas
    df_copia = df_copia.rename(columns={'trimestre': 'ano', 'ano': 'trimestre', 'produto': 'tipo_transacao', 'acesso_atm': 'detalhe_caixa_eletronico'})
    
    # / Reordenando as colunas
    coluna_trimestre = df_copia.pop('trimestre')
    df_copia.insert(1, 'trimestre', coluna_trimestre)
    
    
    # * Coluna data_trimestre: inserindo a data completa
    # / 1. Mapeia qual é o mês e o dia final de cada número de trimestre
    fim_trimestre = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}
    
    # / 2. Junta o ano com o sufixo correspondente do trimestre
    df_copia['data_trimestre'] = (
        df_copia['ano'].astype(str) + df_copia['trimestre'].map(fim_trimestre)
    )
    # / 3. Converte para data
    df_copia['data_trimestre'] = pd.to_datetime(df_copia['data_trimestre']).dt.normalize()
    
    
    
    # / Reordenando coluna data_trimestre
    coluna_data_trimestre = df_copia.pop('data_trimestre')
    df_copia.pop('ano')
    df_copia.insert(0, 'data_trimestre', coluna_data_trimestre)


    # * Coluna canal_acesso: Renomeando termos 
    df_copia['canal_acesso'] = df_copia['canal_acesso'].replace({
        'ATM': 'Caixa Eletrônico',
        'Internet Banking': 'Site do Banco (Internet Banking)',
        'Telefone Celular': 'Aplicativo do Banco (Mobile Banking)'
        })

    # * Coluna
    df_copia['detalhe_caixa_eletronico'] = df_copia['detalhe_caixa_eletronico'].replace({
        'ATM/Acesso aberto/Cartão próprio': 'Caixa Eletrônico / Cartão Próprio',
        'ATM/Acesso aberto/Cartão terceiro': 'Caixa Eletrônico / Banco24Horas (Terceiros)',
        'ATM/Acesso restrito': 'Caixa Eletrônico / Acesso Restrito'
    })
    
    
    display(df_copia['detalhe_caixa_eletronico'].unique())
    display(df_copia.info())
    display(df_copia)
    
     # * Salvando os dados em um arquivo csv
    caminho_csv = os.path.join('..', 'data', 'stg_volumetria_canais.csv')
    
    df_copia.to_csv(caminho_csv, index=False, sep=';', encoding='utf-8-sig', float_format='%.2f')
    print(f'Arquivo salvo com sucesso em: {os.path.abspath(caminho_csv)}')

except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a API: {erro}')



array(['Caixa Eletrônico / Banco24Horas (Terceiros)',
       'Caixa Eletrônico / Cartão Próprio', 'Não aplicável',
       'Caixa Eletrônico / Acesso Restrito'], dtype=object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1744 entries, 0 to 1743
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   data_trimestre            1744 non-null   datetime64[ns]
 1   trimestre                 1744 non-null   int64         
 2   canal_acesso              1744 non-null   object        
 3   tipo_transacao            1744 non-null   object        
 4   detalhe_caixa_eletronico  1744 non-null   object        
 5   qtd_transacoes            1744 non-null   int64         
 6   valor_transacoes          1744 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 95.5+ KB


None

,data_trimestre,trimestre,canal_acesso,tipo_transacao,detalhe_caixa_eletronico,qtd_transacoes,valor_transacoes
0,2020-03-31,1,Caixa Eletrônico,Outras não Financeiras,Caixa Eletrônico / Banco24Horas (Terceiros),304011,0.000000e+00
1,2020-03-31,1,Caixa Eletrônico,Outras não Financeiras,Caixa Eletrônico / Cartão Próprio,7013807,0.000000e+00
2,2020-03-31,1,Site do Banco (Internet Banking),Empréstimos e Financiamentos,Não aplicável,44141850,5.233107e+10
3,2020-03-31,1,Agências,Saque,Não aplicável,60199848,3.098499e+11
4,2020-03-31,1,Caixa Eletrônico,Consulta Saldo/Extrato,Caixa Eletrônico / Banco24Horas (Terceiros),158796677,0.000000e+00
...,...,...,...,...,...,...,...
1739,2026-03-31,1,Aplicativo do Banco (Mobile Banking),Boletos e Convênios,Não aplicável,839191450,8.265450e+11
1740,2026-03-31,1,Centrais de Atendimento,Pix,Não aplicável,135,9.122867e+04
1741,2026-03-31,1,Caixa Eletrônico,Empréstimos e Financiamentos,Caixa Eletrônico / Banco24Horas (Terceiros),65282,7.016561e+07
1742,2026-03-31,1,Caixa Eletrônico,Outras Financeiras,Caixa Eletrônico / Acesso Restrito,1006243,4.923520e+09


Arquivo salvo com sucesso em: c:\Users\mathe\OneDrive\Documentos\Meus Projetos\Análise de Dados\Projeto end-to-end\data\stg_volumetria_canais.csv
